# Processamento de Variáveis Socioeconômicas - Desafio 2

## ZettaLab - Ciência e Governança de Dados

**Objetivo**: Processar dados do IBGE SIDRA para criar variáveis preditoras do modelo de evasão/reprovação escolar.

**Metodologia**: CRISP-DM (Fase 3 - Preparação dos Dados)

**Período**: 2018-2022

**Granularidade**: UF (27 estados)

---

## Variáveis a Processar

| # | Variável | Fonte | Tabela SIDRA |
|---|----------|-------|-------------|
| 1 | Índice de Gini | IBGE/PNAD | 7435 |
| 2 | Taxa de Gravidez Adolescente | IBGE/Registro Civil | 2609 |
| 3 | PIB Total | IBGE/SCR | 5938 |

## Decisões de Projeto

- **Descartada**: Taxa de Analfabetismo (dados faltantes 2020-2021, interpolação inadequada devido à pandemia)
- **Descartada**: Anos de Estudo (estrutura incorreta nos dados baixados)
- **PIB**: Usado como valor total (Mil R$), não per capita (já temos Renda Per Capita)

---

## 1. Setup e Importações

In [1]:
import pandas as pd
import numpy as np
import os

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Caminhos dos diretórios
BASE_DIR = '/home/teodoro/Documents/ZettaLab/ZettaLab-Data'
RAW_DIR = os.path.join(BASE_DIR, 'data', 'Raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'Processed')

print(f"Diretório Raw: {RAW_DIR}")
print(f"Diretório Processed: {PROCESSED_DIR}")

Diretório Raw: /home/teodoro/Documents/ZettaLab/ZettaLab-Data/data/Raw
Diretório Processed: /home/teodoro/Documents/ZettaLab/ZettaLab-Data/data/Processed


In [2]:
# Listar arquivos disponíveis
print("Arquivos Raw:")
for f in os.listdir(RAW_DIR):
    print(f"  - {f}")

print("\nArquivos Processed:")
for f in os.listdir(PROCESSED_DIR):
    print(f"  - {f}")

Arquivos Raw:
  - tx_rend_brasil_regioes_ufs_2021.xlsx
  - RELATORIO_DTB_BRASIL_2024_DISTRITOS.csv
  - tx_rend_brasil_regioes_ufs_2020.xlsx
  - TX_REND_BRASIL_REGIOES_UFS_2018.xlsx
  - desemprego_sindra.csv
  - IDHM.xlsx
  - nascidos_vivos_total_sidra.csv
  - Deslocamento.csv
  - nascidos_vivos_adolescentes_sidra.csv
  - tx_rend_brasil_regioes_ufs_2022.xlsx
  - gini_sidra.csv
  - renda_sintra.csv
  - pib_sidra.csv
  - tx_rend_brasil_regioes_ufs_2019.xlsx

Arquivos Processed:
  - Saneamento.csv
  - indicesEnsino.csv
  - gravidez_adolescente_2018_2022.csv
  - gini_2018_2022.csv
  - dados_modelo_final.csv
  - renda_2018_2022.csv
  - indicadores_educacionais_2018_2022.csv
  - pib_2018_2022.csv
  - desemprego_2018_2022.csv
  - dados_finais_analise.csv
  - Deslocamento_Enriquecido_com_UF_e_Codigos.csv
  - idhm_2018_2022.csv


---

## 2. Processamento do Índice de Gini

**Fonte**: IBGE - PNAD Contínua Anual

**Tabela SIDRA**: 7435

**Descrição**: Índice de Gini do rendimento domiciliar per capita (mede desigualdade de renda)

**Valores**: 0 (igualdade perfeita) a 1 (desigualdade máxima)

In [3]:
# Ler arquivo raw do Gini
gini_raw_path = os.path.join(RAW_DIR, 'gini_sidra.csv')

# Visualizar estrutura do arquivo (primeiras linhas)
with open(gini_raw_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"Linha {i}: {line.strip()}")

Linha 0: ﻿"Tabela 7435 - Índice de Gini do rendimento domiciliar per capita, a preços médios do ano"
Linha 1: "Variável - Índice de Gini do rendimento domiciliar per capita, a preços médios do ano (Índice)"
Linha 2: "Nível";"Cód.";"Unidade da Federação";"Ano"
Linha 3: "Nível";"Cód.";"Unidade da Federação";"2018";"2019";"2020";"2021";"2022"
Linha 4: "UF";"11";"Rondônia";"0,496";"0,472";"0,439";"0,459";"0,447"
Linha 5: "UF";"12";"Acre";"0,558";"0,559";"0,515";"0,539";"0,523"
Linha 6: "UF";"13";"Amazonas";"0,544";"0,566";"0,533";"0,541";"0,509"
Linha 7: "UF";"14";"Roraima";"0,566";"0,580";"0,540";"0,596";"0,547"
Linha 8: "UF";"15";"Pará";"0,562";"0,528";"0,480";"0,529";"0,508"
Linha 9: "UF";"16";"Amapá";"0,547";"0,513";"0,500";"0,530";"0,531"


In [4]:
# Processar Índice de Gini
# O arquivo tem cabeçalho nas linhas 0-3, dados começam na linha 4
# IMPORTANTE: usar header=None para não perder a primeira UF (Rondônia)

gini_df = pd.read_csv(
    gini_raw_path,
    sep=';',
    skiprows=4,  # Pular cabeçalhos do SIDRA (linhas 0-3)
    header=None,  # Não usar primeira linha como header
    encoding='utf-8'
)

print("Colunas originais (índices):", gini_df.columns.tolist())
print(f"\nShape: {gini_df.shape}")
gini_df.head()

Colunas originais (índices): [0, 1, 2, 3, 4, 5, 6, 7]

Shape: (39, 8)


,0,1,2,3,4,5,6,7
0,UF,11,Rondônia,"0,496","0,472","0,439","0,459","0,447"
1,UF,12,Acre,"0,558","0,559","0,515","0,539","0,523"
2,UF,13,Amazonas,"0,544","0,566","0,533","0,541","0,509"
3,UF,14,Roraima,"0,566","0,580","0,540","0,596","0,547"
4,UF,15,Pará,"0,562","0,528","0,480","0,529","0,508"


In [5]:
# Renomear colunas e filtrar apenas UFs
# Estrutura: 0=Nível, 1=Cód, 2=UF, 3=2018, 4=2019, 5=2020, 6=2021, 7=2022

gini_df.columns = ['Nivel', 'Cod', 'UF', '2018', '2019', '2020', '2021', '2022']

# Filtrar apenas linhas onde Nivel = 'UF' (remove notas e fonte)
gini_df = gini_df[gini_df['Nivel'] == 'UF']

# Manter apenas UF e anos
gini_df = gini_df[['UF', '2018', '2019', '2020', '2021', '2022']]

print(f"UFs após filtro: {len(gini_df)}")
print("Colunas após renomear:")
gini_df.head()

UFs após filtro: 27
Colunas após renomear:


,UF,2018,2019,2020,2021,2022
0,Rondônia,"0,496","0,472","0,439","0,459","0,447"
1,Acre,"0,558","0,559","0,515","0,539","0,523"
2,Amazonas,"0,544","0,566","0,533","0,541","0,509"
3,Roraima,"0,566","0,580","0,540","0,596","0,547"
4,Pará,"0,562","0,528","0,480","0,529","0,508"


In [6]:
# Converter de formato wide para long (unpivot)
gini_long = gini_df.melt(
    id_vars=['UF'],
    value_vars=['2018', '2019', '2020', '2021', '2022'],
    var_name='Ano',
    value_name='Indice_Gini'
)

# Converter Ano para inteiro
gini_long['Ano'] = gini_long['Ano'].astype(int)

# Converter valores de string com vírgula para float
# Exemplo: "0,496" -> 0.496
gini_long['Indice_Gini'] = gini_long['Indice_Gini'].str.replace(',', '.').astype(float)

# Ordenar por UF e Ano
gini_long = gini_long.sort_values(['UF', 'Ano']).reset_index(drop=True)

print(f"Shape final: {gini_long.shape}")
print(f"Anos únicos: {sorted(gini_long['Ano'].unique())}")
print(f"UFs únicas: {gini_long['UF'].nunique()}")
gini_long.head(10)

Shape final: (135, 3)
Anos únicos: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
UFs únicas: 27


,UF,Ano,Indice_Gini
0,Acre,2018,0.558
1,Acre,2019,0.559
2,Acre,2020,0.515
3,Acre,2021,0.539
4,Acre,2022,0.523
5,Alagoas,2018,0.550
6,Alagoas,2019,0.527
7,Alagoas,2020,0.510
8,Alagoas,2021,0.526
9,Alagoas,2022,0.498


In [7]:
# Validação dos dados de Gini
print("=== Validação do Índice de Gini ===")
print(f"\nRegistros esperados: 135 (27 UFs x 5 anos)")
print(f"Registros obtidos: {len(gini_long)}")
print(f"\nValores nulos: {gini_long['Indice_Gini'].isna().sum()}")
print(f"\nEstatísticas descritivas:")
print(gini_long['Indice_Gini'].describe())

# Verificar range válido (0 a 1)
min_gini = gini_long['Indice_Gini'].min()
max_gini = gini_long['Indice_Gini'].max()
print(f"\nRange: {min_gini:.3f} - {max_gini:.3f}")
print(f"Range válido (0-1): {'OK' if 0 <= min_gini <= max_gini <= 1 else 'ERRO'}")

=== Validação do Índice de Gini ===

Registros esperados: 135 (27 UFs x 5 anos)
Registros obtidos: 135

Valores nulos: 0

Estatísticas descritivas:
count    135.000000
mean       0.513874
std        0.040774
min        0.412000
25%        0.482500
50%        0.522000
75%        0.545000
max        0.596000
Name: Indice_Gini, dtype: float64

Range: 0.412 - 0.596
Range válido (0-1): OK


In [8]:
# Salvar arquivo processado
gini_output_path = os.path.join(PROCESSED_DIR, 'gini_2018_2022.csv')
gini_long.to_csv(gini_output_path, index=False)

print(f"Arquivo salvo: {gini_output_path}")
print(f"Registros: {len(gini_long)}")

Arquivo salvo: /home/teodoro/Documents/ZettaLab/ZettaLab-Data/data/Processed/gini_2018_2022.csv
Registros: 135


---

## 3. Processamento da Taxa de Gravidez Adolescente

**Fonte**: IBGE - Estatísticas do Registro Civil

**Tabela SIDRA**: 2609

**Descrição**: Percentual de nascidos vivos de mães adolescentes (<20 anos) em relação ao total

**Cálculo**: `(nascidos_menos_15 + nascidos_15_a_19) / total_nascidos * 100`

**Arquivos**:
- `nascidos_vivos_adolescentes_sidra.csv`: nascidos por faixa etária (<15 e 15-19 anos)
- `nascidos_vivos_total_sidra.csv`: total de nascidos vivos

In [9]:
# Visualizar estrutura dos arquivos de nascidos vivos
adolescentes_path = os.path.join(RAW_DIR, 'nascidos_vivos_adolescentes_sidra.csv')
total_path = os.path.join(RAW_DIR, 'nascidos_vivos_total_sidra.csv')

print("=== Arquivo de Adolescentes (primeiras 10 linhas) ===")
with open(adolescentes_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"Linha {i}: {line.strip()}")

=== Arquivo de Adolescentes (primeiras 10 linhas) ===
Linha 0: ﻿"Tabela 2609 - Nascidos vivos, por ano de nascimento, grupos de idade da mãe na ocasião do parto, sexo e lugar de residência da mãe"
Linha 1: "Variável - Nascidos vivos registrados no ano (Pessoas)"
Linha 2: "Nível";"Cód.";"Unidade da Federação";"Sexo";"Ano x Ano de nascimento x Idade da mãe na ocasião do parto"
Linha 3: "Nível";"Cód.";"Unidade da Federação";"Sexo";"2018";;"2019";;"2020";;"2021";;"2022"
Linha 4: "Nível";"Cód.";"Unidade da Federação";"Sexo";"Total";;"Total";;"Total";;"Total";;"Total"
Linha 5: "Nível";"Cód.";"Unidade da Federação";"Sexo";"Menos de 15 anos";"15 a 19 anos";"Menos de 15 anos";"15 a 19 anos";"Menos de 15 anos";"15 a 19 anos";"Menos de 15 anos";"15 a 19 anos";"Menos de 15 anos";"15 a 19 anos"
Linha 6: "UF";"11";"Rondônia";"Total";"225";"4665";"197";"4339";"152";"3861";"175";"3807";"133";"3417"
Linha 7: "UF";"12";"Acre";"Total";"244";"3938";"227";"3771";"163";"3104";"215";"3446";"204";"3090"
Linha

In [10]:
print("=== Arquivo Total (primeiras 10 linhas) ===")
with open(total_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"Linha {i}: {line.strip()}")

=== Arquivo Total (primeiras 10 linhas) ===
Linha 0: ﻿"Tabela 2609 - Nascidos vivos, por ano de nascimento, grupos de idade da mãe na ocasião do parto, sexo e lugar de residência da mãe"
Linha 1: "Variável - Nascidos vivos registrados no ano (Pessoas)"
Linha 2: "Nível";"Cód.";"Unidade da Federação";"Sexo";"Ano x Ano de nascimento x Idade da mãe na ocasião do parto"
Linha 3: "Nível";"Cód.";"Unidade da Federação";"Sexo";"2018";"2019";"2020";"2021";"2022"
Linha 4: "Nível";"Cód.";"Unidade da Federação";"Sexo";"Total";"Total";"Total";"Total";"Total"
Linha 5: "Nível";"Cód.";"Unidade da Federação";"Sexo";"Total";"Total";"Total";"Total";"Total"
Linha 6: "UF";"11";"Rondônia";"Total";"28333";"27246";"25733";"25556";"25032"
Linha 7: "UF";"12";"Acre";"Total";"17266";"17116";"14981";"16125";"15210"
Linha 8: "UF";"13";"Amazonas";"Total";"80890";"80997";"72368";"80208";"78974"
Linha 9: "UF";"14";"Roraima";"Total";"12670";"14398";"12139";"12814";"13290"


In [11]:
# Processar nascidos vivos de adolescentes
# IMPORTANTE: usar header=None para não perder a primeira UF

adolescentes_df = pd.read_csv(
    adolescentes_path,
    sep=';',
    skiprows=6,  # Pular cabeçalhos do SIDRA (6 linhas)
    header=None,  # Não usar primeira linha como header
    encoding='utf-8'
)

print("Colunas originais (índices):", adolescentes_df.columns.tolist())
print(f"Shape: {adolescentes_df.shape}")
adolescentes_df.head(3)

Colunas originais (índices): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Shape: (37, 14)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,UF,11,Rondônia,Total,225.0,4665.0,197.0,4339.0,152.0,3861.0,175.0,3807.0,133.0,3417.0
1,UF,12,Acre,Total,244.0,3938.0,227.0,3771.0,163.0,3104.0,215.0,3446.0,204.0,3090.0
2,UF,13,Amazonas,Total,920.0,16946.0,956.0,16726.0,689.0,14330.0,798.0,15733.0,783.0,14748.0


In [12]:
# Renomear colunas do arquivo de adolescentes e filtrar UFs
# Estrutura: Nivel, Cod, UF, Sexo, 2018_menos15, 2018_15a19, 2019_menos15, 2019_15a19, ...

adolescentes_df.columns = [
    'Nivel', 'Cod', 'UF', 'Sexo',
    '2018_menos15', '2018_15a19',
    '2019_menos15', '2019_15a19',
    '2020_menos15', '2020_15a19',
    '2021_menos15', '2021_15a19',
    '2022_menos15', '2022_15a19'
]

# Filtrar apenas UFs
adolescentes_df = adolescentes_df[adolescentes_df['Nivel'] == 'UF']

# Manter apenas UF e colunas de dados
adolescentes_df = adolescentes_df[[
    'UF',
    '2018_menos15', '2018_15a19',
    '2019_menos15', '2019_15a19',
    '2020_menos15', '2020_15a19',
    '2021_menos15', '2021_15a19',
    '2022_menos15', '2022_15a19'
]]

print(f"UFs: {len(adolescentes_df)}")
adolescentes_df.head()

UFs: 27


,UF,2018_menos15,2018_15a19,2019_menos15,2019_15a19,2020_menos15,2020_15a19,2021_menos15,2021_15a19,2022_menos15,2022_15a19
0,Rondônia,225.0,4665.0,197.0,4339.0,152.0,3861.0,175.0,3807.0,133.0,3417.0
1,Acre,244.0,3938.0,227.0,3771.0,163.0,3104.0,215.0,3446.0,204.0,3090.0
2,Amazonas,920.0,16946.0,956.0,16726.0,689.0,14330.0,798.0,15733.0,783.0,14748.0
3,Roraima,108.0,2353.0,97.0,2612.0,76.0,2058.0,89.0,2212.0,72.0,2232.0
4,Pará,1631.0,30776.0,1609.0,29493.0,1292.0,25902.0,1488.0,27939.0,1457.0,26323.0


In [13]:
# Processar total de nascidos vivos
total_df = pd.read_csv(
    total_path,
    sep=';',
    skiprows=6,  # Pular cabeçalhos do SIDRA
    header=None,  # Não usar primeira linha como header
    encoding='utf-8'
)

print("Colunas originais (índices):", total_df.columns.tolist())
print(f"Shape: {total_df.shape}")

# Renomear colunas e filtrar UFs
total_df.columns = ['Nivel', 'Cod', 'UF', 'Sexo', '2018', '2019', '2020', '2021', '2022']
total_df = total_df[total_df['Nivel'] == 'UF']

# Manter apenas UF e anos
total_df = total_df[['UF', '2018', '2019', '2020', '2021', '2022']]

print(f"UFs: {len(total_df)}")
total_df.head()

Colunas originais (índices): [0, 1, 2, 3, 4, 5, 6, 7, 8]
Shape: (37, 9)
UFs: 27


,UF,2018,2019,2020,2021,2022
0,Rondônia,28333.0,27246.0,25733.0,25556.0,25032.0
1,Acre,17266.0,17116.0,14981.0,16125.0,15210.0
2,Amazonas,80890.0,80997.0,72368.0,80208.0,78974.0
3,Roraima,12670.0,14398.0,12139.0,12814.0,13290.0
4,Pará,145467.0,142255.0,129640.0,138541.0,135164.0


In [14]:
# Calcular taxa de gravidez adolescente para cada ano
# Taxa = (menos_15 + 15_a_19) / total * 100

gravidez_data = []

for ano in ['2018', '2019', '2020', '2021', '2022']:
    for idx, row in adolescentes_df.iterrows():
        uf = row['UF']
        
        # Nascidos de mães adolescentes
        menos_15 = int(row[f'{ano}_menos15'])
        de_15_a_19 = int(row[f'{ano}_15a19'])
        total_adolescentes = menos_15 + de_15_a_19
        
        # Total de nascidos (buscar na outra tabela)
        total_nascidos = int(total_df[total_df['UF'] == uf][ano].values[0])
        
        # Calcular taxa percentual
        taxa = (total_adolescentes / total_nascidos) * 100
        
        gravidez_data.append({
            'UF': uf,
            'Ano': int(ano),
            'Nascidos_Adolescentes': total_adolescentes,
            'Nascidos_Total': total_nascidos,
            'Taxa_Gravidez_Adolescente': round(taxa, 2)
        })

gravidez_df = pd.DataFrame(gravidez_data)

print(f"Shape: {gravidez_df.shape}")
gravidez_df.head(10)

Shape: (135, 5)


,UF,Ano,Nascidos_Adolescentes,Nascidos_Total,Taxa_Gravidez_Adolescente
0,Rondônia,2018,4890,28333,17.26
1,Acre,2018,4182,17266,24.22
2,Amazonas,2018,17866,80890,22.09
3,Roraima,2018,2461,12670,19.42
4,Pará,2018,32407,145467,22.28
5,Amapá,2018,3904,16847,23.17
6,Tocantins,2018,5018,25992,19.31
7,Maranhão,2018,27922,120027,23.26
8,Piauí,2018,9509,49619,19.16
9,Ceará,2018,21496,133899,16.05


In [15]:
# Validação dos dados de Gravidez Adolescente
print("=== Validação da Taxa de Gravidez Adolescente ===")
print(f"\nRegistros esperados: 135 (27 UFs x 5 anos)")
print(f"Registros obtidos: {len(gravidez_df)}")
print(f"\nValores nulos: {gravidez_df['Taxa_Gravidez_Adolescente'].isna().sum()}")
print(f"\nEstatísticas descritivas:")
print(gravidez_df['Taxa_Gravidez_Adolescente'].describe())

# UFs com maior taxa (possível indicador de vulnerabilidade)
print("\nTop 5 UFs com maior taxa média de gravidez adolescente:")
top_uf = gravidez_df.groupby('UF')['Taxa_Gravidez_Adolescente'].mean().sort_values(ascending=False).head(5)
print(top_uf)

=== Validação da Taxa de Gravidez Adolescente ===

Registros esperados: 135 (27 UFs x 5 anos)
Registros obtidos: 135

Valores nulos: 0

Estatísticas descritivas:
count    135.000000
mean      15.591259
std        4.037568
min        7.910000
25%       12.515000
50%       15.550000
75%       18.500000
max       24.220000
Name: Taxa_Gravidez_Adolescente, dtype: float64

Top 5 UFs com maior taxa média de gravidez adolescente:
UF
Acre        22.750
Pará        21.382
Maranhão    21.334
Amazonas    20.990
Amapá       20.562
Name: Taxa_Gravidez_Adolescente, dtype: float64


In [16]:
# Salvar apenas as colunas necessárias para o modelo
gravidez_final = gravidez_df[['UF', 'Ano', 'Taxa_Gravidez_Adolescente']]
gravidez_final = gravidez_final.sort_values(['UF', 'Ano']).reset_index(drop=True)

gravidez_output_path = os.path.join(PROCESSED_DIR, 'gravidez_adolescente_2018_2022.csv')
gravidez_final.to_csv(gravidez_output_path, index=False)

print(f"Arquivo salvo: {gravidez_output_path}")
print(f"Registros: {len(gravidez_final)}")

Arquivo salvo: /home/teodoro/Documents/ZettaLab/ZettaLab-Data/data/Processed/gravidez_adolescente_2018_2022.csv
Registros: 135


---

## 4. Processamento do PIB Total

**Fonte**: IBGE - Sistema de Contas Regionais

**Tabela SIDRA**: 5938

**Descrição**: Produto Interno Bruto a preços correntes

**Unidade**: Mil Reais

**Nota**: Optamos por usar PIB Total ao invés de PIB per capita, pois já temos Renda Per Capita como variável.

In [17]:
# Ler arquivo raw do PIB
pib_raw_path = os.path.join(RAW_DIR, 'pib_sidra.csv')

# Visualizar estrutura
with open(pib_raw_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"Linha {i}: {line.strip()}")

Linha 0: ﻿"Tabela 5938 - Produto interno bruto a preços correntes, impostos, líquidos de subsídios, sobre produtos a preços correntes e valor adicionado bruto a preços correntes total e por atividade econômica, e respectivas participações - Referência 2010"
Linha 1: "Variável - Produto Interno Bruto a preços correntes (Mil Reais)"
Linha 2: "Nível";"Cód.";"Unidade da Federação";"Ano"
Linha 3: "Nível";"Cód.";"Unidade da Federação";"2018";"2019";"2020";"2021";"2022"
Linha 4: "UF";"11";"Rondônia";"44913978";"47091336";"51598741";"58170096";"66795454"
Linha 5: "UF";"12";"Acre";"15331123";"15630017";"16476371";"21374440";"23676136"
Linha 6: "UF";"13";"Amazonas";"100109235";"108181091";"116019139";"131531038";"145140465"
Linha 7: "UF";"14";"Roraima";"13369988";"14292227";"16024276";"18202579";"21095342"
Linha 8: "UF";"15";"Pará";"161349602";"178376984";"215935604";"262904979";"236141874"
Linha 9: "UF";"16";"Amapá";"16795207";"17496661";"18469115";"20099851";"23614291"


In [18]:
# Processar PIB
# IMPORTANTE: usar header=None para não perder a primeira UF
pib_df = pd.read_csv(
    pib_raw_path,
    sep=';',
    skiprows=4,  # Pular cabeçalhos do SIDRA
    header=None,  # Não usar primeira linha como header
    encoding='utf-8'
)

print("Colunas originais (índices):", pib_df.columns.tolist())
print(f"Shape: {pib_df.shape}")
pib_df.head()

Colunas originais (índices): [0, 1, 2, 3, 4, 5, 6, 7]
Shape: (41, 8)


,0,1,2,3,4,5,6,7
0,UF,11,Rondônia,44913978.0,47091336.0,51598741.0,58170096.0,66795454.0
1,UF,12,Acre,15331123.0,15630017.0,16476371.0,21374440.0,23676136.0
2,UF,13,Amazonas,100109235.0,108181091.0,116019139.0,131531038.0,145140465.0
3,UF,14,Roraima,13369988.0,14292227.0,16024276.0,18202579.0,21095342.0
4,UF,15,Pará,161349602.0,178376984.0,215935604.0,262904979.0,236141874.0


In [19]:
# Renomear colunas e filtrar UFs
pib_df.columns = ['Nivel', 'Cod', 'UF', '2018', '2019', '2020', '2021', '2022']
pib_df = pib_df[pib_df['Nivel'] == 'UF']

# Manter apenas UF e anos
pib_df = pib_df[['UF', '2018', '2019', '2020', '2021', '2022']]

print(f"UFs: {len(pib_df)}")
pib_df.head()

UFs: 27


,UF,2018,2019,2020,2021,2022
0,Rondônia,44913978.0,47091336.0,51598741.0,58170096.0,66795454.0
1,Acre,15331123.0,15630017.0,16476371.0,21374440.0,23676136.0
2,Amazonas,100109235.0,108181091.0,116019139.0,131531038.0,145140465.0
3,Roraima,13369988.0,14292227.0,16024276.0,18202579.0,21095342.0
4,Pará,161349602.0,178376984.0,215935604.0,262904979.0,236141874.0


In [20]:
# Converter de formato wide para long
pib_long = pib_df.melt(
    id_vars=['UF'],
    value_vars=['2018', '2019', '2020', '2021', '2022'],
    var_name='Ano',
    value_name='PIB_Total_MilReais'
)

# Converter tipos
pib_long['Ano'] = pib_long['Ano'].astype(int)
pib_long['PIB_Total_MilReais'] = pd.to_numeric(pib_long['PIB_Total_MilReais'], errors='coerce')

# Ordenar
pib_long = pib_long.sort_values(['UF', 'Ano']).reset_index(drop=True)

print(f"Shape final: {pib_long.shape}")
pib_long.head(10)

Shape final: (135, 3)


,UF,Ano,PIB_Total_MilReais
0,Acre,2018,15331123.0
1,Acre,2019,15630017.0
2,Acre,2020,16476371.0
3,Acre,2021,21374440.0
4,Acre,2022,23676136.0
5,Alagoas,2018,54413047.0
6,Alagoas,2019,58963729.0
7,Alagoas,2020,63202349.0
8,Alagoas,2021,76265620.0
9,Alagoas,2022,76065806.0


In [21]:
# Validação dos dados de PIB
print("=== Validação do PIB Total ===")
print(f"\nRegistros esperados: 135 (27 UFs x 5 anos)")
print(f"Registros obtidos: {len(pib_long)}")
print(f"\nValores nulos: {pib_long['PIB_Total_MilReais'].isna().sum()}")
print(f"\nEstatísticas descritivas (em Mil R$):")
print(pib_long['PIB_Total_MilReais'].describe())

# Top 5 UFs por PIB médio
print("\nTop 5 UFs por PIB médio (Mil R$):")
top_pib = pib_long.groupby('UF')['PIB_Total_MilReais'].mean().sort_values(ascending=False).head(5)
for uf, pib in top_pib.items():
    print(f"  {uf}: R$ {pib:,.0f} mil")

=== Validação do PIB Total ===

Registros esperados: 135 (27 UFs x 5 anos)
Registros obtidos: 135

Valores nulos: 0

Estatísticas descritivas (em Mil R$):
count    1.350000e+02
mean     3.044051e+08
std      4.999939e+08
min      1.336999e+07
25%      6.108304e+07
50%      1.422038e+08
75%      3.017740e+08
max      3.130333e+09
Name: PIB_Total_MilReais, dtype: float64

Top 5 UFs por PIB médio (Mil R$):
  São Paulo: R$ 2,557,324,673 mil
  Rio de Janeiro: R$ 879,084,720 mil
  Minas Gerais: R$ 742,771,701 mil
  Rio Grande do Sul: R$ 517,123,463 mil
  Paraná: R$ 511,784,189 mil


In [22]:
# Salvar arquivo processado
pib_output_path = os.path.join(PROCESSED_DIR, 'pib_2018_2022.csv')
pib_long.to_csv(pib_output_path, index=False)

print(f"Arquivo salvo: {pib_output_path}")
print(f"Registros: {len(pib_long)}")

Arquivo salvo: /home/teodoro/Documents/ZettaLab/ZettaLab-Data/data/Processed/pib_2018_2022.csv
Registros: 135


---

## 5. Merge Final - Dataset Consolidado

Combinação de todas as variáveis no dataset final para modelagem.

**Dataset base**: `dados_modelo_final.csv` (já contém indicadores educacionais + IDHM + Desemprego + Renda)

**Variáveis a adicionar**:
- Índice de Gini
- Taxa de Gravidez Adolescente
- PIB Total

In [23]:
# Carregar dataset base existente
base_path = os.path.join(PROCESSED_DIR, 'dados_modelo_final.csv')
df_base = pd.read_csv(base_path)

print("=== Dataset Base ===")
print(f"Shape: {df_base.shape}")
print(f"Colunas: {df_base.columns.tolist()}")
print(f"\nPrimeiros registros:")
df_base.head()

=== Dataset Base ===
Shape: (135, 14)
Colunas: ['UF', 'Ano', 'Taxa_Abandono_Media', 'Taxa_Abandono_EF', 'Taxa_Abandono_EM', 'Taxa_Reprovacao_Media', 'Taxa_Reprovacao_EF', 'Taxa_Reprovacao_EM', 'IDHM', 'Taxa_Desemprego', 'Renda_Per_Capita', 'Indice_Gini', 'Taxa_Gravidez_Adolescente', 'PIB_Total_MilReais']

Primeiros registros:


,UF,Ano,Taxa_Abandono_Media,Taxa_Abandono_EF,Taxa_Abandono_EM,Taxa_Reprovacao_Media,Taxa_Reprovacao_EF,Taxa_Reprovacao_EM,IDHM,Taxa_Desemprego,Renda_Per_Capita,Indice_Gini,Taxa_Gravidez_Adolescente,PIB_Total_MilReais
0,Rondônia,2018,2.20,1.4,3.0,6.35,6.9,5.8,0.730,9.1,1072.0,0.496,17.26,44913978.0
1,Acre,2018,2.95,2.2,3.7,5.70,6.1,5.3,0.733,13.3,868.0,0.558,24.22,15331123.0
2,Amazonas,2018,3.85,2.8,4.9,5.95,6.9,5.0,0.718,14.6,761.0,0.544,22.09,100109235.0
3,Roraima,2018,2.90,2.0,3.8,7.95,7.0,8.9,0.760,14.2,1161.0,0.566,19.42,13369988.0
4,Pará,2018,4.60,3.6,5.6,9.20,11.2,7.2,0.707,10.3,822.0,0.562,22.28,161349602.0


In [24]:
# Carregar as 3 novas variáveis processadas
gini_final = pd.read_csv(os.path.join(PROCESSED_DIR, 'gini_2018_2022.csv'))
gravidez_final = pd.read_csv(os.path.join(PROCESSED_DIR, 'gravidez_adolescente_2018_2022.csv'))
pib_final = pd.read_csv(os.path.join(PROCESSED_DIR, 'pib_2018_2022.csv'))

print(f"Gini: {gini_final.shape}")
print(f"Gravidez: {gravidez_final.shape}")
print(f"PIB: {pib_final.shape}")

Gini: (135, 3)
Gravidez: (135, 3)
PIB: (135, 3)


In [25]:
# Garantir que a coluna Ano está no mesmo tipo em todos os dataframes
df_base['Ano'] = df_base['Ano'].astype(int)
gini_final['Ano'] = gini_final['Ano'].astype(int)
gravidez_final['Ano'] = gravidez_final['Ano'].astype(int)
pib_final['Ano'] = pib_final['Ano'].astype(int)

# Verificar UFs em cada dataset
print("UFs no dataset base:", sorted(df_base['UF'].unique())[:5], "...")
print("UFs no Gini:", sorted(gini_final['UF'].unique())[:5], "...")
print("UFs na Gravidez:", sorted(gravidez_final['UF'].unique())[:5], "...")
print("UFs no PIB:", sorted(pib_final['UF'].unique())[:5], "...")

UFs no dataset base: ['Acre', 'Alagoas', 'Amapá', 'Amazonas', 'Bahia'] ...
UFs no Gini: ['Acre', 'Alagoas', 'Amapá', 'Amazonas', 'Bahia'] ...
UFs na Gravidez: ['Acre', 'Alagoas', 'Amapá', 'Amazonas', 'Bahia'] ...
UFs no PIB: ['Acre', 'Alagoas', 'Amapá', 'Amazonas', 'Bahia'] ...


In [26]:
# Realizar merge das variáveis
# Merge 1: Base + Gini
df_merged = pd.merge(
    df_base,
    gini_final[['UF', 'Ano', 'Indice_Gini']],
    on=['UF', 'Ano'],
    how='left'
)
print(f"Após merge Gini: {df_merged.shape}")

# Merge 2: + Gravidez
df_merged = pd.merge(
    df_merged,
    gravidez_final[['UF', 'Ano', 'Taxa_Gravidez_Adolescente']],
    on=['UF', 'Ano'],
    how='left'
)
print(f"Após merge Gravidez: {df_merged.shape}")

# Merge 3: + PIB
df_merged = pd.merge(
    df_merged,
    pib_final[['UF', 'Ano', 'PIB_Total_MilReais']],
    on=['UF', 'Ano'],
    how='left'
)
print(f"Após merge PIB: {df_merged.shape}")

Após merge Gini: (135, 15)
Após merge Gravidez: (135, 16)
Após merge PIB: (135, 17)


In [27]:
# Validação do dataset final
print("=== Validação do Dataset Final ===")
print(f"\nShape: {df_merged.shape}")
print(f"\nColunas: {df_merged.columns.tolist()}")
print(f"\nValores nulos por coluna:")
print(df_merged.isna().sum())
print(f"\nTotal de valores nulos: {df_merged.isna().sum().sum()}")
print(f"\nTipos de dados:")
print(df_merged.dtypes)

=== Validação do Dataset Final ===

Shape: (135, 17)

Colunas: ['UF', 'Ano', 'Taxa_Abandono_Media', 'Taxa_Abandono_EF', 'Taxa_Abandono_EM', 'Taxa_Reprovacao_Media', 'Taxa_Reprovacao_EF', 'Taxa_Reprovacao_EM', 'IDHM', 'Taxa_Desemprego', 'Renda_Per_Capita', 'Indice_Gini_x', 'Taxa_Gravidez_Adolescente_x', 'PIB_Total_MilReais_x', 'Indice_Gini_y', 'Taxa_Gravidez_Adolescente_y', 'PIB_Total_MilReais_y']

Valores nulos por coluna:
UF                             0
Ano                            0
Taxa_Abandono_Media            0
Taxa_Abandono_EF               0
Taxa_Abandono_EM               0
Taxa_Reprovacao_Media          0
Taxa_Reprovacao_EF             0
Taxa_Reprovacao_EM             0
IDHM                           0
Taxa_Desemprego                0
Renda_Per_Capita               0
Indice_Gini_x                  0
Taxa_Gravidez_Adolescente_x    0
PIB_Total_MilReais_x           0
Indice_Gini_y                  0
Taxa_Gravidez_Adolescente_y    0
PIB_Total_MilReais_y           0
dtype: int64

In [28]:
# Preview do dataset final
print("=== Preview do Dataset Final ===")
df_merged.head(10)

=== Preview do Dataset Final ===


,UF,Ano,Taxa_Abandono_Media,Taxa_Abandono_EF,Taxa_Abandono_EM,Taxa_Reprovacao_Media,Taxa_Reprovacao_EF,Taxa_Reprovacao_EM,IDHM,Taxa_Desemprego,Renda_Per_Capita,Indice_Gini_x,Taxa_Gravidez_Adolescente_x,PIB_Total_MilReais_x,Indice_Gini_y,Taxa_Gravidez_Adolescente_y,PIB_Total_MilReais_y
0,Rondônia,2018,2.20,1.4,3.0,6.35,6.9,5.8,0.730,9.1,1072.0,0.496,17.26,44913978.0,0.496,17.26,44913978.0
1,Acre,2018,2.95,2.2,3.7,5.70,6.1,5.3,0.733,13.3,868.0,0.558,24.22,15331123.0,0.558,24.22,15331123.0
2,Amazonas,2018,3.85,2.8,4.9,5.95,6.9,5.0,0.718,14.6,761.0,0.544,22.09,100109235.0,0.544,22.09,100109235.0
3,Roraima,2018,2.90,2.0,3.8,7.95,7.0,8.9,0.760,14.2,1161.0,0.566,19.42,13369988.0,0.566,19.42,13369988.0
4,Pará,2018,4.60,3.6,5.6,9.20,11.2,7.2,0.707,10.3,822.0,0.562,22.28,161349602.0,0.562,22.28,161349602.0
5,Amapá,2018,3.35,2.4,4.3,10.30,10.4,10.2,0.741,19.8,818.0,0.547,23.17,16795207.0,0.547,23.17,16795207.0
6,Tocantins,2018,1.70,1.1,2.3,7.50,7.4,7.6,0.749,10.5,997.0,0.529,19.31,35666183.0,0.529,19.31,35666183.0
7,Maranhão,2018,3.10,2.3,3.9,6.85,7.7,6.0,0.686,14.4,586.0,0.528,23.26,98179496.0,0.528,23.26,98179496.0
8,Piauí,2018,2.40,1.8,3.0,7.55,9.5,5.6,0.699,12.4,778.0,0.530,19.16,50378418.0,0.530,19.16,50378418.0
9,Ceará,2018,1.05,0.9,1.2,3.00,3.7,2.3,0.739,10.2,818.0,0.547,16.05,155903825.0,0.547,16.05,155903825.0


In [29]:
# Estatísticas descritivas completas
print("=== Estatísticas Descritivas ===")
df_merged.describe()

=== Estatísticas Descritivas ===


,Ano,Taxa_Abandono_Media,Taxa_Abandono_EF,Taxa_Abandono_EM,Taxa_Reprovacao_Media,Taxa_Reprovacao_EF,Taxa_Reprovacao_EM,IDHM,Taxa_Desemprego,Renda_Per_Capita,Indice_Gini_x,Taxa_Gravidez_Adolescente_x,PIB_Total_MilReais_x,Indice_Gini_y,Taxa_Gravidez_Adolescente_y,PIB_Total_MilReais_y
count,135.000000,135.000000,135.000000,135.000000,135.000000,135.000000,135.000000,135.000000,135.000000,135.000000,135.000000,135.000000,1.350000e+02,135.000000,135.000000,1.350000e+02
mean,2020.000000,1.776667,1.348148,2.205185,4.652593,4.702963,4.602222,0.744793,11.326667,1216.192593,0.513874,15.591259,3.044051e+08,0.513874,15.591259,3.044051e+08
std,1.419481,1.114420,0.874331,1.422648,3.148124,3.473308,3.034465,0.042905,3.888448,436.701433,0.040774,4.037568,4.999939e+08,0.040774,4.037568,4.999939e+08
min,2018.000000,0.100000,0.100000,0.100000,0.150000,0.200000,0.100000,0.676000,3.100000,586.000000,0.412000,7.910000,1.336999e+07,0.412000,7.910000,1.336999e+07
25%,2019.000000,0.850000,0.600000,1.100000,1.850000,1.550000,1.850000,0.709000,8.500000,869.000000,0.482500,12.515000,6.108304e+07,0.482500,12.515000,6.108304e+07
50%,2020.000000,1.550000,1.100000,1.700000,4.550000,4.400000,4.500000,0.739000,11.200000,1072.000000,0.522000,15.550000,1.422038e+08,0.522000,15.550000,1.422038e+08
75%,2021.000000,2.550000,2.000000,3.200000,7.025000,6.950000,7.150000,0.774000,14.200000,1492.000000,0.545000,18.500000,3.017740e+08,0.545000,18.500000,3.017740e+08
max,2022.000000,4.600000,3.700000,5.900000,11.900000,13.800000,11.600000,0.859000,20.700000,2802.000000,0.596000,24.220000,3.130333e+09,0.596000,24.220000,3.130333e+09


In [30]:
# Salvar dataset final (sobrescrever o existente)
output_path = os.path.join(PROCESSED_DIR, 'dados_modelo_final.csv')
df_merged.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"DATASET FINAL SALVO: {output_path}")
print(f"{'='*60}")
print(f"\nResumo:")
print(f"  - Registros: {len(df_merged)}")
print(f"  - Colunas: {len(df_merged.columns)}")
print(f"  - Variáveis preditoras: 6")
print(f"  - Variáveis target: 2 (Taxa_Abandono_Media, Taxa_Reprovacao_Media)")
print(f"  - Valores nulos: {df_merged.isna().sum().sum()}")
print(f"\nVariáveis preditoras:")
print(f"  1. IDHM")
print(f"  2. Taxa_Desemprego")
print(f"  3. Renda_Per_Capita")
print(f"  4. Indice_Gini (NOVA)")
print(f"  5. Taxa_Gravidez_Adolescente (NOVA)")
print(f"  6. PIB_Total_MilReais (NOVA)")


DATASET FINAL SALVO: /home/teodoro/Documents/ZettaLab/ZettaLab-Data/data/Processed/dados_modelo_final.csv

Resumo:
  - Registros: 135
  - Colunas: 17
  - Variáveis preditoras: 6
  - Variáveis target: 2 (Taxa_Abandono_Media, Taxa_Reprovacao_Media)
  - Valores nulos: 0

Variáveis preditoras:
  1. IDHM
  2. Taxa_Desemprego
  3. Renda_Per_Capita
  4. Indice_Gini (NOVA)
  5. Taxa_Gravidez_Adolescente (NOVA)
  6. PIB_Total_MilReais (NOVA)


---

## 6. Resumo e Próximos Passos

### Dataset Final

| Aspecto | Valor |
|---------|-------|
| Arquivo | `data/Processed/dados_modelo_final.csv` |
| Registros | 135 (27 UFs × 5 anos) |
| Período | 2018-2022 |
| Variáveis preditoras | 6 |
| Variáveis target | 2 |
| Valores nulos | 0 |

### Variáveis do Modelo

**Targets (variáveis a prever)**:
- `Taxa_Abandono_Media`: Taxa média de abandono escolar (%)
- `Taxa_Reprovacao_Media`: Taxa média de reprovação escolar (%)

**Preditores (variáveis explicativas)**:
1. `IDHM`: Índice de Desenvolvimento Humano Municipal
2. `Taxa_Desemprego`: Taxa de desemprego (%)
3. `Renda_Per_Capita`: Renda per capita (R$)
4. `Indice_Gini`: Índice de desigualdade (0-1)
5. `Taxa_Gravidez_Adolescente`: % de nascidos de mães <20 anos
6. `PIB_Total_MilReais`: PIB estadual (Mil R$)

### Próximos Passos (CRISP-DM)

1. **Fase 4 - Modelagem**: Treinar modelos de ML (Random Forest, XGBoost, etc.)
2. **Fase 5 - Avaliação**: Validar modelos com métricas apropriadas
3. **Fase 6 - Implantação**: Documentar e disponibilizar resultados